In [ ]:
# Databricks notebook source
# DBTITLE 1,SILVER MEMBER GROUP ENROLLMENT PIPELINE
import logging
from datetime import datetime
import os
from pyspark.sql.functions import col, sha2, concat_ws, current_timestamp, to_date

In [ ]:
# Ensure target databases exist under claimspan catalog
spark.sql("CREATE DATABASE IF NOT EXISTS claimspan.silver")
spark.sql("CREATE DATABASE IF NOT EXISTS claimspan.gold")
silverMemGroupTable = "claimspan.silver.silver_member_group"
print(f"Target Silver Member Group Table: {silverMemGroupTable}")


In [ ]:
# Ensure target silver table exists via DDL fallback
spark.sql("""
CREATE TABLE IF NOT EXISTS claimspan.silver.silver_member_group (
    identifier_subscriberID      STRING,
    identifier_beneficiaryID     STRING,
    identifier_cmsContractNumber STRING,
    identifier_groupNumber       STRING,
    extension_groupSuffix        STRING,
    StartDate                    DATE,
    EndDate                      DATE,
    SourceFileID                 BIGINT,
    LoadDateTime                 TIMESTAMP,
    hashKey                      STRING
) USING delta;
""")
if spark.catalog.tableExists("claimspan.silver.silver_member"):
    df_src = spark.table("claimspan.silver.silver_member")
    
    df_grp = df_src.select(
        col("identifier_subscriberID"),
        col("identifier_beneficiaryID"),
        col("extension_coverageProduct_id").alias("identifier_cmsContractNumber"),
        col("identifier_planMemberID").alias("identifier_groupNumber"),
        col("ClientID").alias("extension_groupSuffix"),
        to_date(col("LoadDateTime")).alias("StartDate"),
        to_date(col("deceasedDateTime")).alias("EndDate"),
        col("FileID").cast("bigint").alias("SourceFileID"),
        current_timestamp().alias("LoadDateTime")
    ).distinct()
    
    df_grp_final = df_grp.withColumn(
        "hashKey",
        sha2(concat_ws("|", col("identifier_subscriberID"), col("identifier_beneficiaryID"), col("identifier_cmsContractNumber"), col("identifier_groupNumber"), col("extension_groupSuffix")), 256)
    )
    
    df_grp_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silverMemGroupTable)
    print(f"=== Silver MemberGroup processing complete: {df_grp_final.count()} records created ===")
else:
    print("Silver member table not found, waiting for member ingestion.")